# Pseudospectra of a non-normal operator

The eigenvalues of an operator tell you how perturbations behave *eventually*.
They do not tell you what happens on the way there. When an operator is
**non-normal** — its eigenvectors are not orthogonal — a disturbance can grow by
orders of magnitude before the modal decay eventually wins.

The $\varepsilon$-pseudospectrum makes that visible. It is the set of $z$ where

$$\sigma_{\min}(zI - A) \le \varepsilon,$$

equivalently where the resolvent norm $\|(zI-A)^{-1}\|$ exceeds $1/\varepsilon$.
For a *normal* operator this collapses onto disks of radius $\varepsilon$ around
the eigenvalues. For a non-normal one it bulges far beyond them, and the size of
that bulge is what nonmodal stability analysis measures.

This notebook works through one operator end to end using matrices from the
[NIST Matrix Market](https://math.nist.gov/MatrixMarket/), following the same
path `nonmodal run` takes: infer a region from the spectrum, sample it coarsely,
then refine onto the features.

In [ ]:
import pathlib
import sys

# Reuse the test suite's Matrix Market fetcher: downloads are checksum-verified
# and cached under tests/_data, so re-running this notebook costs nothing.
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'tests'))

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
from matrixmarket import MATRICES, load_dense

from nonmodal import (
    Bounds,
    choose_contour_levels,
    interpolate_to_mesh,
    mirror_conjugates,
    pseudomode_at,
    refine,
    sample_sigmin,
    uniform_points,
)

## The operator

`west0479` is a 479x479 Jacobian from a chemical-engineering model, and a
standard example in the pseudospectra literature. Its *normality defect*
$\|AA^* - A^*A\| / \|A\|^2$ is far from zero, so we should expect the
pseudospectrum to depart substantially from the eigenvalues.

In [ ]:
spec = MATRICES['west0479']
A = load_dense(spec)
eigvals = np.linalg.eigvals(A)

commutator = A @ A.conj().T - A.conj().T @ A
defect = np.linalg.norm(commutator) / np.linalg.norm(A) ** 2

print(f'{spec.name}: {A.shape[0]}x{A.shape[1]}')
print(f'normality defect      : {defect:.3f}   (0 would mean normal)')
print(f'rightmost eigenvalue  : {eigvals.real.max():.4g}')

## Where to sample

The region is always derived from the spectrum — there is no way to hand-pick a
rectangle, because the tool is built to start coarse and zoom in rather than
sweep a chosen box. `Bounds.around_spectrum` is what `nonmodal run` calls, and
`--bounds-pad` is the fraction of the spectral span it leaves around the
eigenvalues.

Sampling needs the Schur factor, not the operator itself: $zI - T$ is
triangular, so each $\sigma_{\min}$ costs a pair of triangular solves rather
than a full factorisation.

In [ ]:
T, Z = scipy.linalg.schur(A, output='complex')
T = np.asarray(T, dtype=np.complex128)
# Z is the unitary of A = Z T Z*. sigma_min is invariant under that change of
# basis -- which is why sampling can work in T -- but singular vectors are not,
# so Z is what carries a pseudomode back to the original basis later on.
Z = np.asarray(Z, dtype=np.complex128)
bounds = Bounds.around_spectrum(eigvals, pad=0.25)

print(f'Re[z] in [{bounds.real_min:9.3f}, {bounds.real_max:9.3f}]')
print(f'Im[z] in [{bounds.imag_min:9.3f}, {bounds.imag_max:9.3f}]')

# load_dense hands back complex128 regardless, so np.isrealobj would say False
# here; the entries are what matter. The pipeline's operator really is float64,
# so there it checks np.isrealobj directly.
has_real_entries = not np.any(A.imag)
print(f'entries are real: {has_real_entries}')

## Conjugate symmetry halves the work

`west0479` is real, so its spectrum is closed under conjugation and therefore so
is its pseudospectrum: $\sigma_{\min}(\bar{z}I - A) = \sigma_{\min}(zI - A)$.
Only $\operatorname{Im} z \ge 0$ needs evaluating; `mirror_conjugates` recovers
the rest for free.

`nonmodal run` decides this with `np.isrealobj` on its own operator, which is
genuinely `float64`. Here `load_dense` returns `complex128` whatever the file
holds, so the cell above inspects the entries instead — same question, asked of
an array that has already been widened.

`sample_sigmin` never mirrors on your behalf: you opt in by choosing which
points to hand it.

In [ ]:
coarse = uniform_points(bounds, nx=25, ny=25)
upper = coarse[coarse.imag >= 0.0]

sig_upper = sample_sigmin(upper, T, nprocs=4)
print(f'{coarse.size} lattice points -> {upper.size} evaluated in the upper half')

# Im z = 0 is always a sampled row, so the axis itself is never stepped over --
# and mirroring must not duplicate it.
on_axis = np.count_nonzero(upper.imag == 0.0)
print(f'{on_axis} of those sit exactly on the real axis')

## Coarse first, then refine

This is the part that matters. Rather than sampling a fine uniform lattice,
spend a small budget uniformly and the rest where the field actually varies.

`refine` triangulates the samples and inserts points into the triangles where a
linear interpolant of $\log_{10}\sigma_{\min}$ is worst — which is exactly the
error that shows up in a contour plot, since contours are drawn by linear
interpolation over that same triangulation.

The budget is a **ceiling, not a promise**: each round inserts at most one point
per triangle, so a large request needs several rounds. A run that finishes short
says so.

In [ ]:
n_initial = upper.size
budget = n_initial + 700

points, sigmin = refine(
    upper, sig_upper,
    lambda batch: sample_sigmin(batch, T, nprocs=4),
    budget=budget, rounds=4)

print(f'initial : {n_initial}')
print(f'added   : {points.size - n_initial} of 700 requested')
print(f'total   : {points.size} evaluations')

Refinement should crowd the region where $\sigma_{\min}$ varies fastest, near
the spectrum, and leave the smooth far field alone. Plotting where the added
points landed makes that concrete.

In [ ]:
added = points[n_initial:]

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(upper.real, upper.imag, s=6, c='0.75', label=f'initial ({n_initial})')
ax.scatter(added.real, added.imag, s=6, c='tab:red',
           label=f'refined ({added.size})')
ax.scatter(eigvals.real, eigvals.imag, s=10, c='k', marker='x',
           label='eigenvalues')

ax.set_xlabel(r'$\mathrm{Re}\,z$')
ax.set_ylabel(r'$\mathrm{Im}\,z$')
ax.set_title('Where refinement spent its budget')
ax.legend(loc='upper left', fontsize=8)
plt.show()

# Quantify the crowding: added points sit closer to the spectrum than the
# uniform start does.
def spectral_distance(pts, eigenvalues):
    """dist(z, spectrum) for each sample point."""
    return np.min(np.abs(pts[:, None] - eigenvalues[None, :]), axis=1)


print(f'mean dist to spectrum, initial : {spectral_distance(upper, eigvals).mean():.3f}')
print(f'mean dist to spectrum, refined : {spectral_distance(added, eigvals).mean():.3f}')

Measured against a dense-SVD reference at equal total cost, this placement
roughly halves interpolation error on `west0479` (2.05x) and is about a wash on
nearly-normal matrices — which is why refinement is off by default. The test
suite pins that comparison in `tests/test_refine.py`, using a Grcar matrix so
the reference is cheap to compute.

## The picture

Now mirror back to the full plane and contour. Contours come straight from a
Delaunay triangulation of the samples — no interpolation onto a mesh, so no
interpolation error where it matters most. This is what `nonmodal plot` does.

Note how far the outer contours extend beyond the spectrum. That gap *is* the
non-normality.

In [ ]:
z, sig = mirror_conjugates(points, sigmin)
levels = choose_contour_levels(sig, min_level=1e-8, nlevels=8)

fig, ax = plt.subplots(figsize=(7.5, 6))
cs = ax.tricontour(z.real, z.imag, np.log10(sig),
                   levels=np.log10(levels), cmap='turbo')
ax.clabel(cs, fmt=lambda v: f'$10^{{{v:.0f}}}$', fontsize=8)
ax.scatter(eigvals.real, eigvals.imag, s=4, c='k', label='eigenvalues')

ax.set_xlabel(r'$\mathrm{Re}\,z$')
ax.set_ylabel(r'$\mathrm{Im}\,z$')
ax.set_title(r'$\varepsilon$-pseudospectrum of west0479')
ax.legend(loc='upper left')
plt.show()

print(f'{z.size} samples after mirroring ({sig.size} values)')

A raster does need a mesh, so the heatmap interpolates — in
$\log_{10}\sigma_{\min}$, never the raw value, which spans several decades
here.

In [ ]:
x, y, log_sig = interpolate_to_mesh(z, sig, mesh=300)

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.pcolormesh(x, y, log_sig, cmap='viridis', shading='auto')
ax.scatter(eigvals.real, eigvals.imag, s=3, c='w')
fig.colorbar(im, ax=ax, label=r'$\log_{10}\sigma_{\min}$')

ax.set_xlabel(r'$\mathrm{Re}\,z$')
ax.set_ylabel(r'$\mathrm{Im}\,z$')
ax.set_title('Interpolated onto a regular mesh')
plt.show()

## How much does non-normality buy?

For *any* matrix, $\sigma_{\min}(zI-A) \le \mathrm{dist}(z, \Lambda(A))$, with
equality exactly when the matrix is normal. So the ratio

$$\frac{\sigma_{\min}(zI-A)}{\mathrm{dist}(z,\Lambda)}$$

is 1 everywhere for a normal operator, and drops below 1 by however much the
non-normality amplifies the resolvent. Its reciprocal is the factor by which a
purely modal argument would *understate* the response.

In [ ]:
ratio = sig / spectral_distance(z, eigvals)
print(f'sigma_min / dist(z, spectrum):  min {ratio.min():.2e}, max {ratio.max():.3f}')
print(f'peak resolvent amplification vs the modal estimate: {1 / ratio.min():.0f}x')

## The mode behind the amplification

The ratio says *how much* the resolvent exceeds the modal estimate. It does not
say *what* gets amplified. That is the **pseudomode**: the right singular vector
$v$ of $zI - A$ belonging to $\sigma_{\min}$, so that

$$\|(A - zI)v\| = \sigma_{\min}(zI - A).$$

It is an approximate eigenvector with a residual you know exactly. Nothing new
is solved to reach it — the inverse iteration that produced $\sigma_{\min}$
also produced this vector, and the sampler simply discarded it.

In [ ]:
# The sampled point where non-normality bought the most.
worst = int(np.argmin(ratio))
z_star = complex(z[worst])
print(f'z* = {z_star:.4f}   sigma_min = {sig[worst]:.4e}')

mode = pseudomode_at(T, Z, z_star)
print(f'residual ||(A - z*I)v||     = {mode.residual:.6e}')
print(f'sigma_min from the sampler  = {sig[worst]:.6e}')
print(f'agreement                   = {abs(mode.residual - sig[worst]) / sig[worst]:.1e} relative')

The comparison that matters is against the eigenvectors. For an eigenvector
$w$ with eigenvalue $\lambda$, $\|(A - zI)w\| = |\lambda - z|$ — so **no**
eigenvector can do better than $\mathrm{dist}(z, \Lambda)$ at this shift. The
pseudomode does, by the whole non-normality factor.

In [ ]:
best_eigenvector = float(np.min(np.abs(z_star - eigvals)))
print(f'best any eigenvector can do: {best_eigenvector:.4e}')
print(f'the pseudomode does:         {mode.residual:.4e}')
print(f'better by:                   {best_eigenvector / mode.residual:.0f}x')

# The mode is a unit vector in the original basis, ready to be written out.
print(f'\n||v|| = {np.linalg.norm(mode.vector):.6f}, length {mode.vector.size}')

On a simulation operator this vector is the thing you actually want to look
at, and `nonmodal pseudomode` writes it as an OFT restart file:

```bash
nonmodal pseudomode --at 5e5-2.4e4j            # one file, best phase
nonmodal pseudomode --at 5e5-2.4e4j --phases 8 # a phase sweep, which animates
```

A complex mode has an arbitrary overall phase, so taking its real part directly
can throw away most of the amplitude; the writer rotates to the phase carrying
the most. `--phases N` then sweeps from there, and because each restart file
records its own index as `t`, the directory reads back as a time series.

That step is skipped here: `west0479` has no field-block layout to scatter into.
Which point to extract at is left to you — nothing searches the plane for you.

## Contrast: a normal operator

`bcsstk01` is a symmetric stiffness matrix, hence normal. The same ratio should
be 1 to within solver tolerance — the pseudospectrum really is just disks around
the eigenvalues, and there is nothing nonmodal to find.

In [ ]:
normal_spec = MATRICES['bcsstk01']
B = load_dense(normal_spec)
b_eigvals = np.linalg.eigvals(B)
B_T = np.asarray(scipy.linalg.schur(B, output='complex')[0], dtype=np.complex128)

b_z = uniform_points(Bounds.around_spectrum(b_eigvals, pad=0.25), nx=20, ny=20)
b_sig = sample_sigmin(b_z, B_T, nprocs=4)

b_ratio = b_sig / spectral_distance(b_z, b_eigvals)
print(f'{normal_spec.name} (symmetric, so normal)')
print(f'  ratio range: {b_ratio.min():.6f} .. {b_ratio.max():.6f}')
print(f'  max deviation from 1: {np.abs(b_ratio - 1).max():.2e}')

## The same thing from the command line

Everything above is what `nonmodal run` does, given the simulation's HDF5
matrices instead of a Matrix Market file:

```bash
# Coarse 25x25 start, then up to 2600 more evaluations placed adaptively.
nonmodal run --grid-nx 25 --grid-ny 25 --refine-points 2600 --nprocs 32

# Rendering is a separate step: no HDF5, no operator, no caches needed, so it
# runs wherever you read the results.
nonmodal plot --output-dir pseudospectrum --min-level 1e-7 --nlevels 16
```

`run` writes the samples flat (`pseudo_z.npy`, `pseudo_sigmin.npy`) alongside
`pseudo_eigvals.npy`, which is why `plot` can draw the eigenvalue overlay from
the output directory alone.

`pseudo_contours` below is the same renderer `nonmodal plot` invokes. Pass
`inline_js=True` to embed plotly.js so the page also opens on a machine without
network access, at the cost of a few MB.

In [ ]:
from nonmodal import pseudo_contours

out = pseudo_contours('.', 'west0479_contours.html', z, sig, eigvals, levels)
print(f'wrote {out} ({pathlib.Path(out).stat().st_size / 1024:.0f} KB)')

## Where to go next

- `sample_sigmin` takes any point set, so you are not restricted to a lattice or
  to `refine`'s strategy. `--grid-npy` feeds the CLI a point set built elsewhere.
- `pseudomode_at` needs both halves of the Schur factorisation. `nonmodal run`
  caches them (`full_reduced_schur.npy`, `full_reduced_schurvecs.npy`), but a
  run predating the second file has to redo the factorisation once.
- Runs are bitwise reproducible: the iterative solver gets a fixed starting
  vector rather than ARPACK's random one.
- Other matrices in `MATRICES` span the range from normal to strongly
  non-normal: `bcsstk01` (0.0), `gre__115` (0.03), `rw136` (0.05),
  `olm100` (0.28), `west0479` (0.63). Re-running this notebook against a milder
  one shows refinement earning much less — which is the honest reason it is
  opt-in.